In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('../../')
import torch
from neural_control.extended_controllers import NNCDIController, FCController
from neural_control.extended_dynamics import SequentialDualSourcing
import plotly.express as px
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
from plotly import graph_objects as go
from sourcing_models.utilities import sample_trajectories_capped_dual_index, \
sample_trajectories_dual_index, sample_trajectories_single_index, sample_trajectories_tailored_base_surge
import time

from neural_control.experiments.helpers import load_experiment_conf

import matplotlib.pyplot as plt
from matplotlib import rcParams
from utilities import magic_random_seed

sourcing_parameters_path = '../../sourcing_models/recursion_input_files/ds1_lr=2_b=95_h=5_u08.in'
experiment_data = load_experiment_conf(sourcing_parameters_path)
total_time = 100
torch.set_default_tensor_type('torch.cuda.FloatTensor')


sourcing_parameters = dict(ce=int(experiment_data.c_e),
                           cr=int(experiment_data.c_r),
                           le=int(experiment_data.l_e),
                           lr=int(experiment_data.l_r),
                           h=int(experiment_data.h),
                           b=int(experiment_data.b),
                           T=total_time,
                           fe = 0,#int(experiment_data.f_e),
                           fr = 0,#int(experiment_data.f_r)

                          )
sourcing_parameters = dict(ce=20,
                           cr=0,
                           le=0,
                           lr=2,
                           h=5,
                           b=495,
                           T=150,
                           fe=100,
                           fr=50
                          )

In [2]:
#Ioannis has some 85 files here wheereas paper has only 95?
sourcing_parameters

{'ce': 20,
 'cr': 0,
 'le': 0,
 'lr': 2,
 'h': 5,
 'b': 495,
 'T': 150,
 'fe': 100,
 'fr': 50}

In [3]:
#!pip install optuna

def training_loop(dynamics, optimizer, optimizer2, minibatch_size):
    dynamics.train()
    all_training_costs = []
    for i in range(300):
        torch.manual_seed(magic_random_seed)
        optimizer.zero_grad()
        optimizer2.zero_grad()

        dynamics.reset(minibatch_size, seed=magic_random_seed)
        nn_context = {}
        total_costs = 0
        for i in range(dynamics.T):
            current_costs, demands, current_inventories, qr, qra, qe, qea, nn_context = dynamics.simulate()
            total_costs = current_costs.mean() + total_costs
        all_training_costs.append(float(total_costs)/dynamics.T)
        total_costs.backward()
        optimizer.step()
        optimizer2.step()

    return all_training_costs

def test_loop(dynamics, minibatch_size):
    dynamics.eval()
    with torch.no_grad():
        all_training_costs = []
        torch.manual_seed(magic_random_seed%5 + 1)
        dynamics.reset(minibatch_size, seed=magic_random_seed)
        nn_context = {}
        total_costs = 0
        for i in range(dynamics.T):
            current_costs, demands, current_inventories, qr, qra, qe, qea, nn_context = dynamics.simulate()
            total_costs = current_costs.mean() + total_costs
        all_training_costs.append(float(total_costs)/dynamics.T)
        return total_costs/dynamics.T

In [4]:
import optuna


possible_activations = {
    'CELU' : torch.nn.CELU(alpha=1),
    # 'RELU' : torch.nn.ReLU(),
    # 'TANH' : torch.nn.Tanh(),
    # 'GELU' : torch.nn.GELU(),
    # 'ELU'  : torch.nn.ELU()
}

def objective(trial: optuna.trial.Trial) -> float:
    minibatch_size = 2**trial.suggest_int('minibatch', 8, 10)
    ### One can experiment with different layer architectures and activations per layer
    nnc_hyperparameters = dict(
        n_hidden_units = []
    )

    ### We use celu non-linearities for the input layer, hidden layers, and output layer
    nnc_hyperparameters['n_activations'] = []


    # We optimize the number of layers, hidden units in each layer and dropouts.
    n_layers = trial.suggest_int("n_layers", 1, 5)
    for i in range(n_layers):
        nnc_hyperparameters['n_hidden_units'].append(2**trial.suggest_int("n_layers_neurons_"+str(i), 1, 8))
        nnc_hyperparameters['n_activations'].append(possible_activations[trial.suggest_categorical("activation", sorted(list(possible_activations.keys())))])

    model = FCController(lr=sourcing_parameters['lr'], 
                       le=sourcing_parameters['le'], 
                       n_hidden_units=nnc_hyperparameters['n_hidden_units'],
                       activations=nnc_hyperparameters['n_activations']
                        )
    I_0 = trial.suggest_int('I_0',1, 8)
    
    dsd = SequentialDualSourcing(model, I_0=I_0, **sourcing_parameters) # Dual Sourcing Dynamics object
    
    lr_model = trial.suggest_float('lr_model',0.001, 0.01)
    lr_inv = trial.suggest_float('lr_inv',0.001, 0.01)

    optimizer = torch.optim.RMSprop(#[dsd.I_0],
                                #list(fcc.parameters()) + [dsd.I_0], 
                                model.parameters(),
                                lr=lr_model
                               )
    optimizer2 = torch.optim.RMSprop([dsd.I_0],
                                    #list(fcc.parameters()), 
                                    lr=lr_inv
                                   )
    training_loop(dsd, optimizer, optimizer2, minibatch_size)
    return -test_loop(dsd, minibatch_size)


In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=1000)

[I 2022-10-13 15:39:54,831] A new study created in memory with name: no-name-84d14884-ad10-4db8-adea-7bf07cab4cb4
[W 2022-10-13 15:41:55,916] Trial 0 failed because of the following error: The value nan is not acceptable.
[W 2022-10-13 15:43:57,191] Trial 1 failed because of the following error: The value nan is not acceptable.
[I 2022-10-13 15:46:05,294] Trial 2 finished with value: -90.37174987792969 and parameters: {'minibatch': 8, 'n_layers': 4, 'n_layers_neurons_0': 3, 'activation': 'CELU', 'n_layers_neurons_1': 3, 'n_layers_neurons_2': 8, 'n_layers_neurons_3': 3, 'I_0': 3, 'lr_model': 0.0037289977477614543, 'lr_inv': 0.0058499888341082785}. Best is trial 2 with value: -90.37174987792969.
[I 2022-10-13 15:48:13,812] Trial 3 finished with value: -2.4251897187529654e+19 and parameters: {'minibatch': 8, 'n_layers': 4, 'n_layers_neurons_0': 7, 'activation': 'CELU', 'n_layers_neurons_1': 7, 'n_layers_neurons_2': 6, 'n_layers_neurons_3': 7, 'I_0': 1, 'lr_model': 0.003160553591000486, 'l

In [ ]:
fine_tuning_iterations = 3000
minibatch_size = 1024
random_seed = 6

all_training_costs = []

t0 = time.time()
for it in range(fine_tuning_iterations):
    torch.manual_seed(magic_random_seed)
    dsd.train()
    
    metaopt.zero_grad()
    optimizer.zero_grad()

    dsd.reset(minibatch_size, seed=magic_random_seed)
    nn_context = {}
    total_costs = 0
    for i in range(dsd.T):
        current_costs, demands, current_inventories, qr, qra, qe, qea, nn_context = dsd.simulate()
        total_costs = current_costs.mean() + total_costs
    all_training_costs.append(float(total_costs)/dsd.T)
    total_costs.backward()
    if it % 100 == 0:
        print(total_costs/dsd.T)def objective(trial: optuna.trial.Trial) -> float:


    return trainer.callback_metrics["val_acc"].item()
    if total_costs < best_loss[0]:
        best_loss[0] = total_costs.detach().cpu().item()
        best_model[0] = deepcopy(fcc.state_dict())
    
    #if it > 1 and all_training_costs[-1] <= 26.4:
    #        break

    optimizer.step()
    metaopt.step()
t1 = time.time()
total = t1-t0
print(total)

In [ ]:
#list(metaopt.parameters())

In [ ]:
list(dsd.named_parameters())

In [ ]:
dsd.I_0

In [ ]:
LearnableOptimizer?

In [ ]:
l2l.optim.LearnableOptimizer??